# Credit Risk Classification (UCI Statlog German Credit)

> **Decision-Support Disclaimer:** This notebook is an educational and decision-support demonstration only. It is not validated for production lending or financial underwriting.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.data_loader import load_german_credit, split_features_target
from src.preprocessing import make_train_val_test_split
from src.train_evaluate import tune_and_train_models, run_error_analysis, run_fairness_audit

df = load_german_credit()
print('Dataset shape:', df.shape)
print('Class distribution:\n', df['class'].value_counts())
df.head(3)

## 1. Stratified Train / Validation / Test Split (60% / 20% / 20%)

In [ ]:
X, y = split_features_target(df)
splits = make_train_val_test_split(X, y, random_state=42)
print(f'Train: {len(splits.X_train)}, Val: {len(splits.X_val)}, Test: {len(splits.X_test)}')

## 2. Train & Compare Logistic Regression, SVM, Decision Tree, and KNN

In [ ]:
trained_models, results = tune_and_train_models(splits)
rows = [{'Model': name, **res['test']} for name, res in results.items()]
pd.DataFrame(rows).drop(columns=['confusion_matrix'])

## 3. Error Analysis & Subgroup Fairness Audit

In [ ]:
best_name = max(results.keys(), key=lambda m: results[m]['validation']['roc_auc'])
best_model = trained_models[best_name]
print('Best validation model:', best_name)
err = run_error_analysis(best_model, splits.X_test, splits.y_test)
pd.DataFrame(err['summary_by_outcome']).T

## 4. Prototype Prediction on a Sample Applicant

In [ ]:
sample = splits.X_test.iloc[[0]]
prob = best_model.predict_proba(sample)[0, 1]
pred = 'High Risk (Bad)' if prob >= 0.5 else 'Low Risk (Good)'
print(f'Predicted Risk: {pred} (Default Probability = {prob:.2%})')